# RQ1. Drivers of Mobile Money Adoption in Ghana

# Problem Statement
Which socioeconomic and financial factors are most strongly associated with mobile money adoption in Ghana? The interim model reached about 97% accuracy, but two predictors (holding any account and having made a digital payment) overlap with the target itself. This notebook removes those near-proxies and builds an honest, explainable driver model.
# Business Context
For the National Financial Inclusion and Development Strategy and for operators, knowing which access and behaviour variables drive adoption tells policy where to aim. We use the pooled Global Findex sample (2,000 respondents, 2021 and 2025 waves).
# Objectives
Remove near-proxy variables; justify the sample size (events per variable); run statistical tests (chi-square, two-proportion z, point-biserial); fit logistic, random-forest, and XGBoost models with stratified cross-validation; explain the best model with SHAP; and reconcile feature importance across four methods.


In [ ]:
# --- Colab setup: install dependencies and load data (run once) ---
import sys, subprocess
def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
try:
    import pmdarima, prophet, shap, xgboost, lightgbm, imblearn  # noqa
except Exception:
    _pip(["pmdarima", "prophet", "shap", "xgboost", "lightgbm", "imbalanced-learn", "seaborn"])

import os
# Clone the repository if the processed data are not already present
if not os.path.exists("data/processed/monthly_series_clean.csv"):
    if not os.path.exists("mobile-money-ghana-forecasting"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bcudjoe/mobile-money-ghana-forecasting.git"])
    os.chdir("mobile-money-ghana-forecasting")

RANDOM_STATE = 42  # fixed seed for reproducibility
os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
print("Setup complete. Working directory:", os.getcwd())


## Analysis
The code below is the exact, tested pipeline that produces the figures in `outputs/figures/` and the metrics in `results/`. It runs top to bottom on a fresh Colab runtime.

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)
import xgboost as xgb
import shap

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
FIG = "outputs/figures"
plt.rcParams.update({"figure.dpi": 200, "savefig.dpi": 200, "font.size": 12,
                     "axes.titlesize": 14, "axes.labelsize": 12})
BLUE, ORANGE, GREY = "#2166AC", "#D6604D", "#888888"

df = pd.read_csv("data/processed/findex_ghana_clean.csv")
res = {}

# ---------------------------------------------------------------------------
# Predictor sets. Near-proxy fields overlap with the target and are dropped.
# ---------------------------------------------------------------------------
NEAR_PROXY = ["has_account", "made_digital_payment"]
DROP = ["resp_id", "adopts_mm"] + NEAR_PROXY
honest_features = [c for c in df.columns if c not in DROP]
# wave is 2021/2025; recode to a clean 0/1 indicator (2025 = 1)
df["wave2025"] = (df["wave"] == 2025).astype(int)
honest_features = [c if c != "wave" else "wave2025" for c in honest_features]
res["near_proxy_dropped"] = NEAR_PROXY
res["honest_features"] = honest_features
res["n_predictors_honest"] = len(honest_features)

y = df["adopts_mm"].astype(int)

# ---------------------------------------------------------------------------
# Minimum sample-size justification (events-per-variable) for the honest model
# ---------------------------------------------------------------------------
n = len(df)
p_adopt = y.mean()
minority_events = int(min(y.sum(), (1 - y).sum()))
epv = minority_events / len(honest_features)
res["sample_size"] = {
    "n": n, "adoption_rate_pct": round(100 * p_adopt, 1),
    "minority_class_events": minority_events,
    "n_predictors": len(honest_features),
    "events_per_variable": round(epv, 1),
    "epv_threshold": 10,
    "meets_epv": bool(epv >= 10),
}
# Power for the two-proportion comparison between waves (Cohen's h)
p1, p2 = df.loc[df.wave == 2021, "adopts_mm"].mean(), df.loc[df.wave == 2025, "adopts_mm"].mean()
h = 2 * np.arcsin(np.sqrt(p2)) - 2 * np.arcsin(np.sqrt(p1))
from statsmodels.stats.power import NormalIndPower
achieved_power = NormalIndPower().power(effect_size=abs(h), nobs1=1000, alpha=0.05, ratio=1.0)
res["sample_size"]["wave_effect_h"] = round(float(h), 3)
res["sample_size"]["achieved_power_wave_test"] = round(float(achieved_power), 3)

# ---------------------------------------------------------------------------
# Outlier analysis on the one continuous predictor (age)
# ---------------------------------------------------------------------------
q1, q3 = df["age"].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
n_out = int(((df["age"] < lo) | (df["age"] > hi)).sum())
res["outliers_age"] = {"q1": float(q1), "q3": float(q3), "iqr": float(iqr),
                       "lower_fence": float(lo), "upper_fence": float(hi),
                       "n_outliers": n_out, "pct": round(100 * n_out / n, 2)}
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
sns.boxplot(x=df["age"], ax=ax[0], color=BLUE)
ax[0].set_title("Age: boxplot (IQR outlier check)")
sns.histplot(df["age"], bins=30, ax=ax[1], color=BLUE)
ax[1].set_title("Age: distribution")
plt.tight_layout(); plt.savefig(f"{FIG}/rq1_age_outliers.png", bbox_inches="tight"); plt.close()

# ---------------------------------------------------------------------------
# Statistical tests (5% level), each with H0/Ha stated in the report
# ---------------------------------------------------------------------------
tests = {}
# Two-proportion z-test: adoption 2021 vs 2025
from statsmodels.stats.proportion import proportions_ztest
cnt = np.array([df.loc[df.wave == 2025, "adopts_mm"].sum(), df.loc[df.wave == 2021, "adopts_mm"].sum()])
nobs = np.array([1000, 1000])
z, pz = proportions_ztest(cnt, nobs)
tests["wave_two_proportion_z"] = {"p2021": round(p1, 3), "p2025": round(p2, 3),
                                  "z": round(float(z), 3), "p_value": float(pz)}
# Chi-square tests of independence: adoption vs categorical drivers
for col in ["income_quintile", "education", "urban", "female", "employed", "owns_mobile", "has_internet"]:
    ct = pd.crosstab(df[col], df["adopts_mm"])
    chi2, pchi, dof, _ = stats.chi2_contingency(ct)
    n_cells = ct.values.sum()
    cramers_v = np.sqrt(chi2 / (n_cells * (min(ct.shape) - 1)))
    tests[f"chi2_{col}"] = {"chi2": round(float(chi2), 2), "dof": int(dof),
                            "p_value": float(pchi), "cramers_v": round(float(cramers_v), 3)}
# Point-biserial: age vs adoption
rpb, ppb = stats.pointbiserialr(df["adopts_mm"], df["age"])
tests["pointbiserial_age"] = {"r": round(float(rpb), 3), "p_value": float(ppb)}
res["statistical_tests"] = tests

# ---------------------------------------------------------------------------
# Baseline logistic WITH vs WITHOUT near-proxy variables (leakage demonstration)
# ---------------------------------------------------------------------------
def fit_logit(features):
    X = df[features].values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
    pipe = Pipeline([("sc", StandardScaler()),
                     ("lr", LogisticRegression(max_iter=1000, class_weight="balanced",
                                               random_state=RANDOM_STATE))])
    pipe.fit(Xtr, ytr)
    pr = pipe.predict(Xte); pp = pipe.predict_proba(Xte)[:, 1]
    return {"accuracy": accuracy_score(yte, pr), "precision": precision_score(yte, pr),
            "recall": recall_score(yte, pr), "f1": f1_score(yte, pr),
            "roc_auc": roc_auc_score(yte, pp)}, pipe, (Xte, yte, pp)

with_proxy_features = [c for c in df.columns if c not in ["resp_id", "adopts_mm", "wave"]] + ["wave2025"]
with_proxy_features = list(dict.fromkeys(with_proxy_features))
m_with, _, _ = fit_logit(with_proxy_features)
m_without, logit_pipe, (Xte_l, yte_l, pp_l) = fit_logit(honest_features)
res["logit_with_proxy"] = {k: round(v, 4) for k, v in m_with.items()}
res["logit_without_proxy"] = {k: round(v, 4) for k, v in m_without.items()}

# Standardised logistic coefficients (honest model, full-data refit for ranking)
sc = StandardScaler().fit(df[honest_features].values)
lr_full = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
lr_full.fit(sc.transform(df[honest_features].values), y)
logit_coef = dict(zip(honest_features, lr_full.coef_[0]))

# ---------------------------------------------------------------------------
# Tree models with stratified CV + SHAP (honest features)
# ---------------------------------------------------------------------------
X = df[honest_features].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "RandomForest": RandomForestClassifier(n_estimators=400, max_depth=6, class_weight="balanced",
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": xgb.XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                                 subsample=0.9, colsample_bytree=0.9,
                                 scale_pos_weight=float((1 - p_adopt) / p_adopt) ** 0,  # ~1, balanced-ish
                                 eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
}
model_metrics = {}
for name, mdl in models.items():
    auc_cv = cross_val_score(mdl, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    mdl.fit(Xtr, ytr)
    pr = mdl.predict(Xte); pp = mdl.predict_proba(Xte)[:, 1]
    model_metrics[name] = {"cv_roc_auc_mean": round(float(auc_cv.mean()), 4),
                           "cv_roc_auc_std": round(float(auc_cv.std()), 4),
                           "accuracy": round(accuracy_score(yte, pr), 4),
                           "precision": round(precision_score(yte, pr), 4),
                           "recall": round(recall_score(yte, pr), 4),
                           "f1": round(f1_score(yte, pr), 4),
                           "roc_auc": round(roc_auc_score(yte, pp), 4)}
# also logistic honest CV auc for the comparison table
lr_cv = cross_val_score(Pipeline([("sc", StandardScaler()),
                                  ("lr", LogisticRegression(max_iter=1000, class_weight="balanced",
                                   random_state=RANDOM_STATE))]), X, y, cv=cv, scoring="roc_auc")
model_metrics["Logistic"] = {"cv_roc_auc_mean": round(float(lr_cv.mean()), 4),
                             "cv_roc_auc_std": round(float(lr_cv.std()), 4),
                             **{k: round(v, 4) for k, v in m_without.items()}}
res["model_metrics_honest"] = model_metrics

best_name = max(["RandomForest", "XGBoost"], key=lambda k: model_metrics[k]["cv_roc_auc_mean"])
best = models[best_name]
res["best_rq1_model"] = best_name

# SHAP on best tree model
explainer = shap.TreeExplainer(best)
shap_vals = explainer.shap_values(Xte)
if isinstance(shap_vals, list):          # older API: list per class
    shap_vals = shap_vals[1]
shap_vals = np.asarray(shap_vals)
if shap_vals.ndim == 3:                  # newer API: (n, features, classes)
    shap_vals = shap_vals[:, :, 1]
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
shap_imp = dict(zip(honest_features, mean_abs_shap))

# SHAP summary (beeswarm) + bar
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_vals, Xte, feature_names=honest_features, show=False, plot_size=(9, 6))
plt.tight_layout(); plt.savefig(f"{FIG}/rq1_shap_beeswarm.png", bbox_inches="tight"); plt.close()
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_vals, Xte, feature_names=honest_features, plot_type="bar", show=False)
plt.tight_layout(); plt.savefig(f"{FIG}/rq1_shap_bar.png", bbox_inches="tight"); plt.close()

# Permutation + impurity importance on RF for cross-model table
rf = models["RandomForest"]
perm = permutation_importance(rf, Xte, yte, n_repeats=20, random_state=RANDOM_STATE, scoring="roc_auc")
perm_imp = dict(zip(honest_features, perm.importances_mean))
impurity_imp = dict(zip(honest_features, rf.feature_importances_))

# Cross-model importance comparison (rank each method)
imp_df = pd.DataFrame({
    "logit_abs_coef": {k: abs(v) for k, v in logit_coef.items()},
    "rf_impurity": impurity_imp,
    "permutation": perm_imp,
    "mean_abs_shap": shap_imp,
})
for col in imp_df.columns:
    imp_df[col + "_rank"] = imp_df[col].rank(ascending=False)
imp_df["avg_rank"] = imp_df[[c for c in imp_df.columns if c.endswith("_rank")]].mean(axis=1)
imp_df = imp_df.sort_values("avg_rank")
imp_df.to_csv("results/rq1_cross_model_importance.csv")
res["cross_model_importance_top"] = imp_df.head(8).round(4).reset_index().rename(columns={"index": "feature"}).to_dict("records")
res["logit_coef_signed"] = {k: round(v, 3) for k, v in sorted(logit_coef.items(), key=lambda x: -abs(x[1]))}

# ROC curve figure (best model + logistic)
pp_best = best.predict_proba(Xte)[:, 1]
fpr_b, tpr_b, _ = roc_curve(yte, pp_best)
fpr_l, tpr_l, _ = roc_curve(yte_l, pp_l)
plt.figure(figsize=(7, 6))
plt.plot(fpr_b, tpr_b, color=BLUE, lw=2, label=f"{best_name} (AUC={roc_auc_score(yte, pp_best):.3f})")
plt.plot(fpr_l, tpr_l, color=ORANGE, lw=2, ls="--", label=f"Logistic (AUC={roc_auc_score(yte_l, pp_l):.3f})")
plt.plot([0, 1], [0, 1], color=GREY, ls=":", lw=1)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("RQ1 ROC curves, near-proxy variables removed")
plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig(f"{FIG}/rq1_roc.png", bbox_inches="tight"); plt.close()

# Confusion matrix for best model
cm = confusion_matrix(yte, best.predict(Xte))
plt.figure(figsize=(5.5, 4.8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-adopter", "Adopter"], yticklabels=["Non-adopter", "Adopter"])
plt.title(f"RQ1 confusion matrix, {best_name}"); plt.ylabel("Actual"); plt.xlabel("Predicted")
plt.tight_layout(); plt.savefig(f"{FIG}/rq1_confusion.png", bbox_inches="tight"); plt.close()

# Driver ranking bar (mean abs SHAP, top 10)
top = imp_df.head(10).index.tolist()
plt.figure(figsize=(9, 6))
vals = [shap_imp[f] for f in top]
sns.barplot(x=vals, y=top, color=BLUE)
plt.xlabel("Mean |SHAP value|"); plt.title(f"RQ1 driver ranking ({best_name}, SHAP)")
plt.tight_layout(); plt.savefig(f"{FIG}/rq1_driver_ranking.png", bbox_inches="tight"); plt.close()

with open("results/rq1_results.json", "w") as fp:
    json.dump(res, fp, indent=2, default=str)

# Console summary
print("=== RQ1 SUMMARY ===")
print("Near-proxy dropped:", NEAR_PROXY)
print("EPV:", res["sample_size"]["events_per_variable"], "meets>=10:", res["sample_size"]["meets_epv"])
print("Wave z-test p:", tests["wave_two_proportion_z"]["p_value"])
print("Logit WITH proxy acc/auc:", m_with["accuracy"], m_with["roc_auc"])
print("Logit WITHOUT proxy acc/auc:", round(m_without["accuracy"],3), round(m_without["roc_auc"],3))
for k, v in model_metrics.items():
    print(f"  {k}: CV-AUC={v['cv_roc_auc_mean']} test-AUC={v.get('roc_auc')}")
print("Best RQ1 model:", best_name)
print("Top drivers (avg rank):", imp_df.head(6).index.tolist())
print("Age outliers:", res["outliers_age"]["n_outliers"])
print("DONE")

## Observations on RQ1
- After removing `has_account` and `made_digital_payment`, logistic accuracy falls from about 97% to about 77% and the ROC-AUC to about 0.85, which is the removal of leakage rather than a loss of skill.
- The strongest drivers by average rank across four importance methods are mobile phone ownership, formal saving, the 2025 wave, education, and income.
- Every gradient tested is statistically significant (for example, mobile ownership: chi-square about 394, p < .001), so the visual gradients are backed by formal tests.
- The events-per-variable ratio is about 46, far above the floor of 10, so the sample is more than adequate.
